# Exploratory notebook for lyrics pipeline

Much of the code in this notebook was used to test functions during development and therefore cannot be reproduced using the latest implementation in src/.

In [2]:
# autoreload every imported modules if their source files have changed.
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

print("cwd:", Path.cwd())
print("src exists:", Path("src").exists())

cwd: /Users/chs/Documents/LLM-zoomcamp/Capstone/LyricLens
src exists: True


In [1]:
from src.download_hot100 import download_hot100
download_hot100()

Downloaded hot-100 data to data/raw/hot-100-current.csv.


In [2]:
from src.clean_hot100 import clean_hot100_song, filter_hot100_song
hot100_song_df = clean_hot100_song(save=False)
filtered_hot100_song_df = filter_hot100_song(hot100_song_df, start_wk="2020-01-01", end_wk="2021-01-01", save=False)

32676 songs recorded in data/raw/hot-100-current.csv.
575 songs on Billboard Hot-100 from 20200101 to 20210101.


In [5]:
from src.search_lyrics import search_batch_lyrics

In [6]:
filtered_hot100_lyrics_df = search_batch_lyrics(filtered_hot100_song_df)

  0%|          | 0/575 [00:00<?, ?it/s]

Saved checkpoints of 100 songs at data/processed/checkpoint.parquet.
Saved checkpoints of 200 songs at data/processed/checkpoint.parquet.
Saved checkpoints of 300 songs at data/processed/checkpoint.parquet.
Saved checkpoints of 400 songs at data/processed/checkpoint.parquet.
Saved checkpoints of 500 songs at data/processed/checkpoint.parquet.


In [7]:
len(filtered_hot100_lyrics_df)

575

In [8]:
filtered_hot100_lyrics_df.head()

,title,performer,chart_weeks,wks_on_chart,peak_pos,plain_lyrics
349,20/20,Lil Tjay,[2020-01-18 00:00:00],1,94,"I feel like the greatest, only been at this fo..."
356,21,Polo G,"[2020-05-30 00:00:00, 2020-06-06 00:00:00, 202...",8,62,"Decorate your block with red tape, foenem slid..."
368,24,Money Man Featuring Lil Baby,"[2020-08-29 00:00:00, 2020-09-05 00:00:00, 202...",14,49,"Yo, Nflated, spice that bitch up\n\nBurnin' on..."
391,3 Headed Goat,Lil Durk Featuring Lil Baby & Polo G,"[2020-05-23 00:00:00, 2020-05-30 00:00:00, 202...",16,43,Aviator\n\nThese ain't no Guess jeans\nI dropp...
408,33,Polo G,[2020-05-30 00:00:00],1,93,High off ecstasy and that codeine what I'm sip...


In [9]:
import pandas as pd

In [11]:
sum(filtered_hot100_lyrics_df["plain_lyrics"].apply(lambda x: x is pd.NA))

0

In [13]:
sum(filtered_hot100_lyrics_df["plain_lyrics"].apply(lambda x: x == "<error>"))

0

In [68]:
from src.search_lyrics import retry_batch_error

In [69]:
filtered_hot100_lyrics_0error_df = retry_batch_error(filtered_hot100_lyrics_df)

All request errors have been resolved.


In [70]:
sum(filtered_hot100_lyrics_0error_df["plain_lyrics"].isna())

10

In [18]:
filtered_hot100_lyrics_0error_df.head()

,title,performer,chart_weeks,wks_on_chart,peak_pos,plain_lyrics
349,20/20,Lil Tjay,[2020-01-18 00:00:00],1,94,"I feel like the greatest, only been at this fo..."
356,21,Polo G,"[2020-05-30 00:00:00, 2020-06-06 00:00:00, 202...",8,62,"Decorate your block with red tape, foenem slid..."
368,24,Money Man Featuring Lil Baby,"[2020-08-29 00:00:00, 2020-09-05 00:00:00, 202...",14,49,"Yo, Nflated, spice that bitch up\n\nBurnin' on..."
391,3 Headed Goat,Lil Durk Featuring Lil Baby & Polo G,"[2020-05-23 00:00:00, 2020-05-30 00:00:00, 202...",16,43,Aviator\n\nThese ain't no Guess jeans\nI dropp...
408,33,Polo G,[2020-05-30 00:00:00],1,93,High off ecstasy and that codeine what I'm sip...


In [17]:
filtered_hot100_lyrics_0error_df.tail()

,title,performer,chart_weeks,wks_on_chart,peak_pos,plain_lyrics
32510,Young Wheezy,NAV With Gunna,[2020-11-21 00:00:00],1,84,"Yeah\n(Wheezy outta here)\nYoung Wheezy, Young..."
32611,Yummy,Justin Bieber,"[2020-01-18 00:00:00, 2020-01-25 00:00:00, 202...",15,2,"Yeah, you got that yummy-yum\nThat yummy-yum, ..."
32628,Zoo York,Lil Tjay Featuring Fivio Foreign & Pop Smoke,[2020-05-23 00:00:00],1,65,"Grr, ayy (Ah)\nNah, bow-bow-bow-bow (Woo)\nBow..."
32644,g n f (Give No Fxk),"Migos, Young Thug & Travis Scott",[2020-02-29 00:00:00],1,48,None
32653,ily,Surf Mesa Featuring Emilee,"[2020-06-06 00:00:00, 2020-06-13 00:00:00, 202...",28,23,"I love you, baby\nAnd if it's quite all right\..."


In [71]:
filtered_hot100_lyrics_0error_df["plain_lyrics"].apply(lambda x: x is None).sum()

np.int64(10)

In [72]:
filtered_hot100_lyrics_NONE_df = filtered_hot100_lyrics_0error_df[filtered_hot100_lyrics_0error_df["plain_lyrics"].apply(lambda x: x is None)]
filtered_hot100_lyrics_NONE_df

,title,performer,chart_weeks,wks_on_chart,peak_pos,plain_lyrics
1357,All These N**gas,King Von Featuring Lil Durk,[2020-11-21 00:00:00],1,77,None
4090,Can I,Kehlani Featuring Tory Lanez,[2020-05-23 00:00:00],1,50,None
5342,Crazy Story 2.0,King Von Featuring Lil Durk,[2020-11-21 00:00:00],1,81,None
6220,Dive Bar,Garth Brooks & Blake Shelton,"[2020-02-08 00:00:00, 2020-02-15 00:00:00, 202...",5,78,None
9468,Go Stupid,Polo G Featuring NLE Choppa & Stunna 4 Vegas,"[2020-02-29 00:00:00, 2020-03-07 00:00:00, 202...",9,60,None
14153,Iris,Phoebe & Maggie,[2020-11-28 00:00:00],1,57,None
24342,Snitchin,Pop Smoke Featuring Quavo & Future,[2020-07-18 00:00:00],1,54,None
28901,Tragic,The Kid LAROI Featuring YoungBoy Never Brok Ag...,[2020-11-21 00:00:00],1,76,None
29695,WHAT TO DO?,JACKBOYS Featuring Don Toliver,"[2020-01-11 00:00:00, 2020-01-18 00:00:00]",2,56,None
32644,g n f (Give No Fxk),"Migos, Young Thug & Travis Scott",[2020-02-29 00:00:00],1,48,None


In [73]:
import requests

from src.search_lyrics import LRCLIB_SEARCH_URL, HEADERS
def search_song_q_lyrics(track_name, artist_name):
    result = None
    track_name = track_name.strip().title()
    artist_name = (
        artist_name.strip()
        .title()
        .replace(" Featuring ", " ")
        .replace(" With ", " ")
        .replace(" X ", " ")
        .replace(" & ", " ")
    )  # format featuring artists, and collaborators
    q = f"{track_name} {artist_name}"
    response = requests.get(
                LRCLIB_SEARCH_URL,
                params={"q": q},
                headers=HEADERS,
                timeout=(3, 5),  # wait for 3s until connect timeout, 5s until read timeout
            )
    results = response.json()
    if results:
        result = results[0].get("plainLyrics")
    return result


In [74]:
filtered_hot100_lyrics_NONE_df.iloc[0]

title                      All These N**gas
performer       King Von Featuring Lil Durk
chart_weeks           [2020-11-21 00:00:00]
wks_on_chart                              1
peak_pos                                 77
plain_lyrics                           None
Name: 1357, dtype: object

In [75]:
result = search_song_q_lyrics("All These N**gas", "King Von Featuring Lil Durk")
result

'DJ on the beat so it\'s a banger, yeah\n\nAll these niggas actin\' like they with that shit\nI ain\'t stuntin\' these niggas (let\'s get it)\nLamb\' truck sittin\' so low had to crouch my back\nFuck around, had to sit on my pistol\nHow you gon\' back door, nigga, you love for a little bit of clout?\nThat\'s a shame on niggas (shame on niggas)\n\nHow you gon\' sit in my car\nTryna play Lil\' Pump like we ain\'t two dangerous niggas?\nBro got trial, he lost faith\nHe looked up when they changed the verdict (let\'s get it)\nRobbers goofy, Stains pay me for a song\nDon\'t clear the verse (don\'t clear the verse)\nWe don\'t go off names\nI don\'t care who they is, we go off murders (we go off murders)\nThis your first time buying that ZaZa, I need 4K for the sherbet\n\nHe a pussy, I know niggas in his hood, that boy a hoe (that boy a hoe)\nThat pussy dookie\nI be fuckin\' his main bitch, and he don\'t know (and he don\'t know)\nWe got .45 drums, everytime they see me, I\'m on go (bow-bow)\

In [31]:
for i in range(len(filtered_hot100_lyrics_NONE_df)):
    result = search_song_q_lyrics(filtered_hot100_lyrics_NONE_df.iloc[i]["title"], filtered_hot100_lyrics_NONE_df.iloc[i]["performer"])
    if result is None:
        print(i, filtered_hot100_lyrics_NONE_df.iloc[i], sep="\n")

5
title                            Iris
performer             Phoebe & Maggie
chart_weeks     [2020-11-28 00:00:00]
wks_on_chart                        1
peak_pos                           57
plain_lyrics                     None
Name: 14153, dtype: object
7
title                                                      Tragic
performer       The Kid LAROI Featuring YoungBoy Never Brok Ag...
chart_weeks                                 [2020-11-21 00:00:00]
wks_on_chart                                                    1
peak_pos                                                       76
plain_lyrics                                                 None
Name: 28901, dtype: object
9
title                        g n f (Give No Fxk)
performer       Migos, Young Thug & Travis Scott
chart_weeks                [2020-02-29 00:00:00]
wks_on_chart                                   1
peak_pos                                      48
plain_lyrics                                None
Name: 32644, dtype: ob

In [46]:
from src.search_lyrics import primary_performer

In [35]:
print(filtered_hot100_lyrics_NONE_df.iloc[5]["title"], filtered_hot100_lyrics_NONE_df.iloc[5]["performer"], sep="\n")

Iris
Phoebe & Maggie


In [52]:
primary_performer("Phoebe & Maggie")

'Phoebe*'

In [53]:
result = search_song_q_lyrics("Iris", "Phoebe*")
result

"Sometimes I think I'm a killer\nI scared you in your house\nI even scared myself by talking\nAbout Dahmer on your couch\nBut I can't sleep next to a body\nEven harmless in death\nPlus, I'm pretty sure I'd miss you\nAnd faking sleep to count your breath\n\nCan the killer in me tame the fire in you?\nOh, is there nothing left to do for us?\nI am sick of the chase but I'm hungry for blood\nAnd there's nothing I can do\n\nBut when I'm sick and tired\nAnd when my mind is barely there\nWhen a machine keeps me alive\nAnd I'm losing all my hair\nI hope you kiss my rotten head\nAnd pull the plug\nKnow that I've burned every playlist\nAnd I've given all my love\n\nCan the killer in me tame the fire in you?\nI know there's something waiting for us\nI am sick of the chase but I'm stupid in love\nAnd there's nothing I can do\n\nAnd there's nothing I can do"

In [36]:
print(filtered_hot100_lyrics_NONE_df.iloc[7]["title"], filtered_hot100_lyrics_NONE_df.iloc[7]["performer"], sep="\n")

Tragic
The Kid LAROI Featuring YoungBoy Never Brok Again & Internet Money


In [54]:
primary_performer("The Kid LAROI Featuring YoungBoy Never Brok Again & Internet Money")

'The Kid Laroi*'

In [55]:
result = search_song_q_lyrics("Tragic" , "The Kid LAROI*")
result

"Yeah, yeah\n\nI remember times when I ain't have shit\nNo food in my crib, now I live lavish (Oh, no-no-no)\nThey say they love me to my face, but they can't stand it\nThem hoes ain't fuck with me back then, now they attracted\nFN attached to my bro and he gon' blast it\nDon't tell me what I've been through, you don't know\nMade it out the mud, with or without you, I'm good\nDid it for my lil' bro, just to show him one day you could\nI did this for my mama 'cause I told her one day I would\n\nMy life been tragic, thank you for askin'\nI've seen too many of my fam end up in caskets\nI've seen too many of my friends turned into ashes\nWish I still had them\nLight a blunt, put it to the sky, I'm tryna get high (High)\nThey won't stop, look me in my eyes, tell me it's alright (Alright)\nHad to cut plenty people off, money on my mind\nI can't fuck with 'em, I can't trust nobody but I (Yeah)\n\nI remember times when I ain't have shit (Yeah)\nNo food in my crib, now I live lavish\nThey say t

In [56]:
print(filtered_hot100_lyrics_NONE_df.iloc[9]["title"], filtered_hot100_lyrics_NONE_df.iloc[9]["performer"], sep="\n")

g n f (Give No Fxk)
Migos, Young Thug & Travis Scott


In [62]:
primary_performer("Migos, Young Thug & Travis Scott")

'Migos*'

In [59]:
result = search_song_q_lyrics("g n f (Give No Fxk)" , "Migos, Young Thug*")
result

In [60]:
result = search_song_q_lyrics("g n f (Give No Fxk)" , "Migos*")
result

In [61]:
result = search_song_q_lyrics("g n f (Give No Fxk)" , "")
result

In [63]:
from src.search_lyrics import retry_batch_none

In [64]:
filtered_hot100_lyrics_0none_df = retry_batch_none(filtered_hot100_lyrics_0error_df)

10 songs have None lyrics before retry.


  0%|          | 0/10 [00:00<?, ?it/s]

Retry finished. 1 Nones remain.


In [65]:
filtered_hot100_lyrics_0none_0error_df = retry_batch_errors(filtered_hot100_lyrics_0none_df)

All request errors have been resolved.
Retry finished. 0 errors remain.


In [66]:
filtered_hot100_lyrics_0none_0error_df["plain_lyrics"].isna().sum()

np.int64(1)

In [67]:
a = True
~a

/var/folders/3q/grt8y4b96hl0dyrqfcj1vyg00000gn/T/ipykernel_84713/3979167406.py:2: DeprecationWarning: Bitwise inversion '~' on bool is deprecated and will be removed in Python 3.16. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  ~a


-2

---

Observation: A lot of the problemetic songs have "Featuring" in its preformer.

In [37]:
from src.search_lyrics import search_song_lyrics

In [ ]:
result = search_song_lyrics(title="$ex Appeal", performer="Baby Keem Featuring Too $hort")

In [31]:
result = search_song_lyrics(title="$ex Appeal", performer="Baby Keem Featuring Too $hort")
result

In [43]:
"Baby Keem Featuring Too $hort".split(" Featuring ")

['Baby Keem', 'Too $hort']

In [33]:
result = search_song_lyrics(title="$ex Appeal", performer="Baby Keem")
result

'(Play with me, play with me)\n(Play with me, pla-play with me) up all night, baby\n(Play with me, play with me)\n(Play with me, pla-play with me) it\'s $hort Dog\n\n(Play with me, play with me)\n(Play with me, pla-play with me) bright lights\n(Play with me, play with me)\n(Play with me, pla-play) Ca$ino\n\nI\'m in the mood (I\'m in the mood), we in Miami (Miami)\nI met a freak (I met a freak) from Cincinnati (from Cincinnati)\nShe\'s five foot four, Margiela Tabis\nUnder the hood (under the hood), she got no panties (bitch)\nHer body smooth (her body smooth), I know she want me\nParty animal, I\'m in the moment and I\'m irrational\nSpur of the moment, she get magical\n\nToo much sex appeal (ah)\nToo much sex appeal (ah)\nToo much sex appeal\nToo much sex appeal (ah)\n\nToo much sex appeal (ah)\nToo much sex appeal (ah)\nI want you, can\'t you tell?\nToo much sex appeal (ah)\nTell me (yeah)\n\nLove the way you talk, you drop that amazin\' (amazin\')\nFirst name basis, call me, "Baby" (

"Featuring ..." in the performer is causing trouble.

The following can solve the problem:

str.split(" Featuring ")[0].strip()

In [40]:
mask = filtered_hot100_lyrics_NONE_df["performer"].str.contains("Featuring", na=False)
sum(mask)

123

In [44]:
from src.search_lyrics import retry_batch_none

In [47]:
filtered_hot100_lyrics_0none_df = retry_batch_none(filtered_hot100_lyrics_0error_df)
filtered_hot100_lyrics_0none_df

135 songs have None lyrics before retry.


  0%|          | 0/135 [00:00<?, ?it/s]

Retry finished. 12 Nones remain.


,title,performer,chart_weeks,wks_on_chart,peak_pos,plain_lyrics
349,20/20,Lil Tjay,[2020-01-18 00:00:00],1,94,"I feel like the greatest, only been at this fo..."
356,21,Polo G,"[2020-05-30 00:00:00, 2020-06-06 00:00:00, 202...",8,62,"Decorate your block with red tape, foenem slid..."
368,24,Money Man Featuring Lil Baby,"[2020-08-29 00:00:00, 2020-09-05 00:00:00, 202...",14,49,"Yo, Nflated, spice that bitch up\n\nBurnin' on..."
391,3 Headed Goat,Lil Durk Featuring Lil Baby & Polo G,"[2020-05-23 00:00:00, 2020-05-30 00:00:00, 202...",16,43,Aviator\n\nThese ain't no Guess jeans\nI dropp...
408,33,Polo G,[2020-05-30 00:00:00],1,93,High off ecstasy and that codeine what I'm sip...
...,...,...,...,...,...,...
32510,Young Wheezy,NAV With Gunna,[2020-11-21 00:00:00],1,84,None
32611,Yummy,Justin Bieber,"[2020-01-18 00:00:00, 2020-01-25 00:00:00, 202...",15,2,"Yeah, you got that yummy-yum\nThat yummy-yum, ..."
32628,Zoo York,Lil Tjay Featuring Fivio Foreign & Pop Smoke,[2020-05-23 00:00:00],1,65,"Grr, ayy (Ah)\nNah, bow-bow-bow-bow (Woo)\nBow..."
32644,g n f (Give No Fxk),"Migos, Young Thug & Travis Scott",[2020-02-29 00:00:00],1,48,None


In [49]:
filtered_hot100_lyrics_0none_df.duplicated(subset=["title", "performer"]).sum()

np.int64(0)

In [50]:
sum(filtered_hot100_lyrics_0none_df["plain_lyrics"].isna())

12

In [51]:
sum(filtered_hot100_lyrics_0none_df["plain_lyrics"].apply(lambda x: x is None))

12

In [52]:
filtered_hot100_lyrics_NONE_df = filtered_hot100_lyrics_0none_df[filtered_hot100_lyrics_0none_df["plain_lyrics"].isna()]
filtered_hot100_lyrics_NONE_df

,title,performer,chart_weeks,wks_on_chart,peak_pos,plain_lyrics
4883,Come & Go,Juice WRLD x Marshmello,"[2020-07-25 00:00:00, 2020-08-01 00:00:00, 202...",20,2,None
6220,Dive Bar,Garth Brooks & Blake Shelton,"[2020-02-08 00:00:00, 2020-02-15 00:00:00, 202...",5,78,None
14153,Iris,Phoebe & Maggie,[2020-11-28 00:00:00],1,57,None
15450,La Jeepeta,Nio Garcia x Anuel AA x Myke Towers x Brray x ...,"[2020-08-22 00:00:00, 2020-08-29 00:00:00, 202...",10,93,None
15478,La Santa,Bad Bunny X Daddy Yankee,"[2020-03-14 00:00:00, 2020-03-21 00:00:00]",2,53,None
16143,Life's A Mess,Juice WRLD X Halsey,"[2020-07-18 00:00:00, 2020-07-25 00:00:00, 202...",6,9,None
19471,No Dribble,DaBaby x Stunna 4 Vegas,[2020-08-15 00:00:00],1,92,None
20867,Past Life,Trevor Daniel x Selena Gomez,"[2020-07-11 00:00:00, 2020-07-25 00:00:00, 202...",5,77,None
21958,Real Shit,Juice WRLD x benny blanco,[2020-12-19 00:00:00],1,72,None
23941,Sigues Con El,Arcangel x Sech,"[2020-04-25 00:00:00, 2020-05-23 00:00:00, 202...",3,78,None


"x", "X", "With", "&" are causing problems. It is possible that the querry can not process multiple performers in search.

In [56]:
result = search_song_lyrics(title="Come & Go", performer="Juice WRLD Marshmello")
result

"Whoa\nUh\nOh\n(Mello made it right)\n\nI try to be everythin' that I can\nBut sometimes I come out as being nothin'\nI try to be everything that I can\nBut sometimes I come out as bein' nothin'\n\nI pray to God that He make me a better man, uh\nMaybe one day, I'ma stand for somethin'\nI'm thankin' God that He made you part of the plan\nI guess I ain't go through all that Hell for nothin'\nI'm always fuckin' up and wreckin' shit, it seems like I perfected it\nI offer you my love, I hope you take it like some medicine\nYou tell me, ain't nobody better than me, I think that there's better than me\nHope you see the better in me, always end up betterin' me\n\nI don't wanna ruin this one\nThis type of love don't always come and go\nI don't wanna ruin this one\nThis type of love don't always come and go\nI don't wanna ruin this one\nThis type of love don't always come and go\nI don't wanna ruin this one\nThis type of love don't always come and go\n\nI don't wanna ruin this one\nThis type of 

In [62]:
result = search_song_lyrics(title="Dive Bar", performer="Garth Brooks Blake Shelton")
result

In [58]:
result = search_song_lyrics(title="Dive Bar", performer="Garth Brooks")
result

"Well, turn that bottle up and drink it\nCrank that jukebox up and Hank it\nBartender, pour another round\nHere's to our best bad decisions\nSituation, no conditions\nOh, and memories we all need to drown\n\nSo fill your cup and raise it up\nJump in and join the club\nAnd float this whisky river reservoir\nWe're gonna spend the weekend\nIn the deep end\nOf a dive bar\n\n'Cause up in here you're not the only\nLoved and left her lost and lonely one\nWho's ever swam against the tide (woo)\nThinkin' this is your oasis\nIt's the safest of the places (hey, I like that)\nThat a broken heart can find to hide\n\nSo here's a toast coast to coast\nWith a big ol' adios\nTo wishes wasted on them fallin' stars\nWe're gonna spend the weekend\nIn the deep end\nOf a dive bar\n\nYeah, it's just chapter after chapter\nOf happy never after\nBut that's just the way the story goes\nFor some bar stool believers\nWear our heart out on our sleevers\nJust goin' where the neon glows\n\nYeah, it's just chapter af

In [60]:
filtered_hot100_lyrics_NONE_df.iloc[3]["performer"]

'Nio Garcia x Anuel AA x Myke Towers x Brray x Juanka'

In [15]:
result = search_song_lyrics(title="La Jeepeta", performer="Nio Garcia Anuel AA Myke Towers Brray Juanka")
result

NameError: name 'search_song_lyrics' is not defined

In [64]:
result = search_song_lyrics(title="Young Wheezy", performer="NAV Gunna")
result

"Yeah\n(Wheezy outta here)\nYoung Wheezy, Young Wheezy, Young Wheezy, Young Wheezy\nYoung Wheezy, Young Wheezy, Young Wheezy, Young Wheezy\nIt's Young Gunna, Wunna, Young NAV, and Young Wheezy\nYoung Gunna, Young Wunna, Young NAV, and Young Wheezy (slatt)\nYoung Wheezy, Young Wheezy, Young Wheezy, Young Wheezy\nYoung Wheezy, Young Wheezy, Young Wheezy, Young Wheezy\n\nOh, they fresh out a coffin, don't know why they talkin'\nGot power like Austin, they cap with a gown\nThese rappers is exhaustin', that ho shit gon' cost 'em\nThey wet like a faucet, I look like a fountain\nGet Texas like Austin, in LA, we golfin'\nCan't take no more losses and we steady countin'\nI put 'em in office and didn't get a crown\nI'm drenched like a dolphin, the freshest in town\n\nRacks in, walk in, Maxfield and I'm splurgin' again\nMixin' up Crush like it's juice and the gin\nStick in her pussy, it's loosenin', yeah (ooh)\nI went to Houston with Travis, I'm lit (it's lit)\nNiggas send threats and ain't nobod

In [66]:
filtered_hot100_lyrics_NONE_df.iloc[-1]["title"]

'g n f (Give No Fxk)'

In [67]:
result = search_song_lyrics(title="g n f (Give No Fxk)", performer="Young Thug Travis Scott")	
result

The following can solve the problem:

str.replace(" With ", " ").replace(" x ", " ").replace(" X ", " ").replace(" & ", " ")

What about unify treatment with " Featuring "?

str.replace(" Featuring ", " ").replace(" With ", " ").replace(" x ", " ").replace(" X ", " ").replace(" & ", " ")

In [68]:
result = search_song_lyrics(title="$ex Appeal", performer="Baby Keem Too $hort")
result

'(Play with me, play with me)\n(Play with me, pla-play with me) up all night, baby\n(Play with me, play with me)\n(Play with me, pla-play with me) it\'s $hort Dog\n\n(Play with me, play with me)\n(Play with me, pla-play with me) bright lights\n(Play with me, play with me)\n(Play with me, pla-play) Ca$ino\n\nI\'m in the mood (I\'m in the mood), we in Miami (Miami)\nI met a freak (I met a freak) from Cincinnati (from Cincinnati)\nShe\'s five foot four, Margiela Tabis\nUnder the hood (under the hood), she got no panties (bitch)\nHer body smooth (her body smooth), I know she want me\nParty animal, I\'m in the moment and I\'m irrational\nSpur of the moment, she get magical\n\nToo much sex appeal (ah)\nToo much sex appeal (ah)\nToo much sex appeal\nToo much sex appeal (ah)\n\nToo much sex appeal (ah)\nToo much sex appeal (ah)\nI want you, can\'t you tell?\nToo much sex appeal (ah)\nTell me (yeah)\n\nLove the way you talk, you drop that amazin\' (amazin\')\nFirst name basis, call me, "Baby" (

YEAH!

In [73]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [74]:
filtered_hot100_lyrics_0none_df = retry_batch_none(filtered_hot100_lyrics_0error_df)
filtered_hot100_lyrics_0none_df

135 songs have None lyrics before retry.


  0%|          | 0/135 [00:00<?, ?it/s]

Retry finished. 12 Nones remain.


,title,performer,chart_weeks,wks_on_chart,peak_pos,plain_lyrics
349,20/20,Lil Tjay,[2020-01-18 00:00:00],1,94,"I feel like the greatest, only been at this fo..."
356,21,Polo G,"[2020-05-30 00:00:00, 2020-06-06 00:00:00, 202...",8,62,"Decorate your block with red tape, foenem slid..."
368,24,Money Man Featuring Lil Baby,"[2020-08-29 00:00:00, 2020-09-05 00:00:00, 202...",14,49,"Yo, Nflated, spice that bitch up\n\nBurnin' on..."
391,3 Headed Goat,Lil Durk Featuring Lil Baby & Polo G,"[2020-05-23 00:00:00, 2020-05-30 00:00:00, 202...",16,43,Aviator\n\nThese ain't no Guess jeans\nI dropp...
408,33,Polo G,[2020-05-30 00:00:00],1,93,High off ecstasy and that codeine what I'm sip...
...,...,...,...,...,...,...
32510,Young Wheezy,NAV With Gunna,[2020-11-21 00:00:00],1,84,None
32611,Yummy,Justin Bieber,"[2020-01-18 00:00:00, 2020-01-25 00:00:00, 202...",15,2,"Yeah, you got that yummy-yum\nThat yummy-yum, ..."
32628,Zoo York,Lil Tjay Featuring Fivio Foreign & Pop Smoke,[2020-05-23 00:00:00],1,65,"Grr, ayy (Ah)\nNah, bow-bow-bow-bow (Woo)\nBow..."
32644,g n f (Give No Fxk),"Migos, Young Thug & Travis Scott",[2020-02-29 00:00:00],1,48,None


In [75]:
sum(filtered_hot100_lyrics_0none_df["plain_lyrics"].isna())

12

In [76]:
filtered_hot100_lyrics_NONE_df = filtered_hot100_lyrics_0none_df[filtered_hot100_lyrics_0none_df["plain_lyrics"].isna()]
filtered_hot100_lyrics_NONE_df

,title,performer,chart_weeks,wks_on_chart,peak_pos,plain_lyrics
4883,Come & Go,Juice WRLD x Marshmello,"[2020-07-25 00:00:00, 2020-08-01 00:00:00, 202...",20,2,None
6220,Dive Bar,Garth Brooks & Blake Shelton,"[2020-02-08 00:00:00, 2020-02-15 00:00:00, 202...",5,78,None
14153,Iris,Phoebe & Maggie,[2020-11-28 00:00:00],1,57,None
15450,La Jeepeta,Nio Garcia x Anuel AA x Myke Towers x Brray x ...,"[2020-08-22 00:00:00, 2020-08-29 00:00:00, 202...",10,93,None
15478,La Santa,Bad Bunny X Daddy Yankee,"[2020-03-14 00:00:00, 2020-03-21 00:00:00]",2,53,None
16143,Life's A Mess,Juice WRLD X Halsey,"[2020-07-18 00:00:00, 2020-07-25 00:00:00, 202...",6,9,None
19471,No Dribble,DaBaby x Stunna 4 Vegas,[2020-08-15 00:00:00],1,92,None
20867,Past Life,Trevor Daniel x Selena Gomez,"[2020-07-11 00:00:00, 2020-07-25 00:00:00, 202...",5,77,None
21958,Real Shit,Juice WRLD x benny blanco,[2020-12-19 00:00:00],1,72,None
23941,Sigues Con El,Arcangel x Sech,"[2020-04-25 00:00:00, 2020-05-23 00:00:00, 202...",3,78,None


---

In [21]:
result_df.at[203,"title"]

"(There's No Place Like) Home For The Holidays (1954)"

In [22]:
result_df.at[204,"title"]

"(There's No Place Like) Home For The Holidays (1959)"

The problem of same song still presists...

In [28]:
import pandas as pd

In [29]:
a = result_df["plain_lyrics"].apply(lambda x: x is pd.NA)
a

13       True
16       True
22       True
24       True
203      True
         ... 
32664    True
32667    True
32668    True
32669    True
32671    True
Name: plain_lyrics, Length: 4115, dtype: bool

In [30]:
pending_indices = result_df.index[a]
pending_indices


Index([   13,    16,    22,    24,   203,   204,   259,   266,   268,   281,
       ...
       32657, 32658, 32661, 32662, 32663, 32664, 32667, 32668, 32669, 32671],
      dtype='int64', length=4115)

In [35]:
import requests
from src.search_lyrics import HEADERS, search_song_lyrics

In [32]:
index= pending_indices[0]
index

np.int64(13)

In [33]:
row = result_df.loc[index]
row

title                              $ex Appeal
performer       Baby Keem Featuring Too $hort
chart_weeks             [2026-03-07 00:00:00]
wks_on_chart                                1
peak_pos                                   69
plain_lyrics                             <NA>
Name: 13, dtype: object

In [40]:
result_df.at[index, "plain_lyrics"] = search_song_lyrics(row["title"], row["performer"])
result_df.head()

Failed (LRCLib): $ex Appeal by Baby Keem Featuring Too $hort: HTTPSConnectionPool(host='lrclib.net', port=443): Max retries exceeded with url: /api/search?track_name=%24Ex+Appeal&artist_name=Baby+Keem+Featuring+Too+%24Hort (Caused by NewConnectionError("HTTPSConnection(host='lrclib.net', port=443): Failed to establish a new connection: [Errno 61] Connection refused"))


,title,performer,chart_weeks,wks_on_chart,peak_pos,plain_lyrics
13,$ex Appeal,Baby Keem Featuring Too $hort,[2026-03-07 00:00:00],1,69,<error>
16,'98 Braves,Morgan Wallen,"[2023-03-18 00:00:00, 2023-03-25 00:00:00, 202...",7,27,<NA>
22,'Til You Can't,Cody Johnson,"[2021-10-23 00:00:00, 2021-10-30 00:00:00, 202...",29,18,<NA>
24,'Tis The Damn Season,Taylor Swift,"[2020-12-26 00:00:00, 2021-01-02 00:00:00]",2,39,<NA>
203,(There's No Place Like) Home For The Holidays ...,Perry Como With Mitchell Ayers And His Orchestra,[2024-01-06 00:00:00],1,50,<NA>


In [42]:
row = result_df.loc[pending_indices[1]]
row

title                                                  '98 Braves
performer                                           Morgan Wallen
chart_weeks     [2023-03-18 00:00:00, 2023-03-25 00:00:00, 202...
wks_on_chart                                                    7
peak_pos                                                       27
plain_lyrics                                                 <NA>
Name: 16, dtype: object

In [50]:
result_df.at[pending_indices[1], "plain_lyrics"] = search_song_lyrics(row["title"], row["performer"])
result_df.at[pending_indices[1], "plain_lyrics"]

"I remember sittin' at that house\nLivin' room couch\nThinkin' no way them boys wouldn't win\nBetween them, big three pitchers\nAndrew and chipper\nIt was gonna be hard to keep up with the Jones'\nBut as fate would have it\nThat Atlanta magic\nGot put out by them damn Padres\n\nI guess destiny\nAin't always meant to be\nKinda like you and me that day\n\nWe got close\nBut close doesn't cut it\nHad a good run and end up with nothin'\nBut a three by five\nThat you hide in a drawer\nWe swung for the fences\nAnd came up short\n\nYea, you win some and lose some\nIt ain't always home runs\nAnd that's just the way life plays\nIf we were a team and love was a game\nWe'd been the '98 Braves\n\nHad that whole town believin'\nDamn girl, I even had that talk to your dad man to man\nBut just like that season\nGirl, you and me didn't end with a ring on a hand\n\nWe got close\nBut close doesn't cut it\nHad a good run and end up with nothin'\nBut a three by five\nThat you hide in a drawer\nWe swung for

In [51]:
result_df.head()

,title,performer,chart_weeks,wks_on_chart,peak_pos,plain_lyrics
13,$ex Appeal,Baby Keem Featuring Too $hort,[2026-03-07 00:00:00],1,69,<error>
16,'98 Braves,Morgan Wallen,"[2023-03-18 00:00:00, 2023-03-25 00:00:00, 202...",7,27,I remember sittin' at that house\nLivin' room ...
22,'Til You Can't,Cody Johnson,"[2021-10-23 00:00:00, 2021-10-30 00:00:00, 202...",29,18,<NA>
24,'Tis The Damn Season,Taylor Swift,"[2020-12-26 00:00:00, 2021-01-02 00:00:00]",2,39,<NA>
203,(There's No Place Like) Home For The Holidays ...,Perry Como With Mitchell Ayers And His Orchestra,[2024-01-06 00:00:00],1,50,<NA>


In [46]:
result_df.loc[index]

title                              $ex Appeal
performer       Baby Keem Featuring Too $hort
chart_weeks             [2026-03-07 00:00:00]
wks_on_chart                                1
peak_pos                                   69
plain_lyrics                          <error>
Name: 13, dtype: object

In [3]:
import requests


def search_lyrics(title: str, performer: str) -> dict | None:
    """Search LRCLIB for lyrics records.

    Args:
        title: The title of the song for which to search lyrics.
        performer: The performer of the song.

    Returns:
        The top matching LRCLIB lyrics record.

    Raises:
        ValueError: If title or performer is empty.
        requests.HTTPError: If LRCLIB returns an unsuccessful response.
        requests.RequestException: If the request failed.
    """
    track_name = title.strip()
    artist_name = performer.strip()

    if not track_name:
        raise ValueError("track_name cannot be empty.")

    if not artist_name:
        raise ValueError("artist_name cannot be empty.")
    
    try:
        response = requests.get(
                LRCLIB_SEARCH_URL,
                params={
                    "track_name": track_name,
                    "artist_name": artist_name,
                },
                headers=HEADERS,
                timeout=(3,5), # wait for 3s to establish connection, 5s for the server to respond
            )
        response.raise_for_status()
        
    except requests.HTTPError as e:
        status_code = e.response.status_code

        if status_code == 404:
            print("Song not found.")
        elif status_code == 429:
            print("Rate limit exceeded.")
        elif status_code >= 500:
            print("Server error.")
        else:
            print(f"HTTP error: {status_code}")
        raise
    
    except requests.RequestException as e: # Timeout, ConnectionError, TooManyRedirects, etc.
        print(f"Request failed: {e}") 
        raise   
    
    results = response.json()

    return results[0] if results else None

In [4]:
result = search_lyrics("Blinding Lights", "The Weeknd")

In [5]:
result

{'id': 37015057,
 'name': 'Blinding Lights',
 'trackName': 'Blinding Lights',
 'artistName': 'The Weeknd',
 'albumName': 'The Com-Bil-ation',
 'duration': 202.0,
 'instrumental': False,
 'plainLyrics': "Yeah\n\nI've been tryna call\n\nI've been on my own for long enough\nMaybe you can show me how to love, maybe\n\nI'm going through withdrawals\n\nYou don't even have to do too much\nYou can turn me on with just a touch, baby\nI look around and\nSin City's cold and empty (oh)\nNo one's around to judge me (oh)\nI can't see clearly when you're gone\nI said, ooh, I'm blinded by the lights\nNo, I can't sleep until I feel your touch\n\nI said, ooh, I'm drowning in the night\nOh, when I'm like this, you're the one I trust\n(Hey, hey, hey)\n\nI'm running out of time\n\n'Cause I can see the sun light up the sky\nSo I hit the road in overdrive, baby, oh\nThe city's cold and empty (oh)\nNo one's around to judge me (oh)\nI can't see clearly when you're gone\nI said, ooh, I'm blinded by the lights\n

MVP: just plainLyrics

In [6]:
result['plainLyrics']

"Yeah\n\nI've been tryna call\n\nI've been on my own for long enough\nMaybe you can show me how to love, maybe\n\nI'm going through withdrawals\n\nYou don't even have to do too much\nYou can turn me on with just a touch, baby\nI look around and\nSin City's cold and empty (oh)\nNo one's around to judge me (oh)\nI can't see clearly when you're gone\nI said, ooh, I'm blinded by the lights\nNo, I can't sleep until I feel your touch\n\nI said, ooh, I'm drowning in the night\nOh, when I'm like this, you're the one I trust\n(Hey, hey, hey)\n\nI'm running out of time\n\n'Cause I can see the sun light up the sky\nSo I hit the road in overdrive, baby, oh\nThe city's cold and empty (oh)\nNo one's around to judge me (oh)\nI can't see clearly when you're gone\nI said, ooh, I'm blinded by the lights\nNo, I can't sleep until I feel your touch\n\nI said, ooh, I'm drowning in the night\nOh, when I'm like this, you're the one I trust\n\nI'm just walking by to let you know (by to let you know)\nI can nev

In [ ]:
import pandas as pd
from src.clean_hot100 import OUTPUT_DIR
from pathlib import Path

HOT100_WITH_LYRICS_OUTPUT = Path(OUTPUT_DIR) / "hot100-song&lyrics.parquet"

def add_lyrics(songs_df: pd.DataFrame) -> pd.DataFrame:
    result_df = songs_df.copy()

    for i, (_, row) in enumerate(songs_df.iterrows(), start=1):
        try:
            result = search_lyrics(row["title"], row["performer"])
            result_df["plain_lyrics"] = result.get("plainLyrics", None)
        except Exception:
            result_df["plain_lyrics"] = None

        if i % 100 == 0:
            result_df.to_parquet(HOT100_WITH_LYRICS_OUTPUT, index=False)
            print(f"Checkpoint saved after {i} songs.")


    return result_df